# 07 — GFF3 Annotations

**Purpose:** Demonstrate how to retrieve and work with phage gene annotations stored
in GFF3 format using the `GFF3Retriever` class.

| | |
|---|---|
| **Expected inputs** | GFF3 files at `/data/processed/gff3/` and index at `gff3_index.json` |
| **Outputs** | Examples saved under `<results>/07_gff3_annotations/` |
| **Companion notebooks** | `01` database QC · `02` sequence retrieval |

## What you will learn

1. How to initialize `GFF3Retriever` and explore the index
2. Retrieve raw GFF3 content for a specific phage
3. Parse GFF3 lines into structured data (CDS features)
4. Batch retrieval and listing examples
5. Integration with phage metadata from DuckDB

---
## Setup

In [1]:
import sys
import os
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pandas as pd

from pbi import GFF3Retriever

# Results directory
results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
results_dir = results_root / '07_gff3_annotations'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Results dir: {results_dir}')

Results dir: /results/07_gff3_annotations


---
## 1. Initialize GFF3Retriever

The `GFF3Retriever` uses a pre-built JSON index for O(1) lookup of phage annotations.
The index maps each `phage_id` to its location in the per-database GFF3 files.

In [2]:
# Default paths (container mode)
GFF3_DIR = Path('/data/processed/gff3')
GFF3_INDEX = GFF3_DIR / 'gff3_index.json'

# Fallback to local paths if not in container
if not GFF3_INDEX.exists():
    GFF3_DIR = project_root / 'data' / 'processed' / 'gff3'
    GFF3_INDEX = GFF3_DIR / 'gff3_index.json'

if not GFF3_INDEX.exists():
    print(f'WARNING: GFF3 index not found at {GFF3_INDEX}')
    print('Run the pipeline first to generate GFF3 files and index.')
else:
    gff3 = GFF3Retriever(str(GFF3_DIR), str(GFF3_INDEX))
    print(f'GFF3Retriever initialized')
    print(f'  Index: {GFF3_INDEX}')
    print(f'  GFF3 dir: {GFF3_DIR}')

GFF3Retriever initialized
  Index: /data/processed/gff3/gff3_index.json
  GFF3 dir: /data/processed/gff3


---
## 2. Index Overview

The index provides statistics about the GFF3 collection: total phage count
and breakdown by source database.

In [3]:
stats = gff3.stats()

print(f'Total phages with GFF3 annotations: {stats["total_phages"]:,}')
print(f'\nSources ({len(stats["sources"])} databases):')
for source, count in sorted(stats['sources'].items(), key=lambda x: -x[1]):
    print(f'  {source:15s} {count:>8,}')

Total phages with GFF3 annotations: 1,166,106

Sources (21 databases):
  MetaVR           225,269
  GOV2             195,699
  MGV              189,680
  GPD              142,809
  TemPhD            66,823
  ELGV              56,716
  CHVD              44,935
  URPC              44,633
  TYMEFLIES         44,192
  OVD               37,184
  GVD               31,402
  UHGV              21,176
  BAPS              18,590
  OPD               14,649
  IGVD              10,021
  GSV                4,518
  STV                4,065
  HPGC               3,872
  PhagesDB           3,754
  VMGC               3,485
  SVD                2,634


---
## 3. List Available Sources

You can list all source databases or filter phages by source.

In [4]:
# List all source databases
source_databases = sorted(stats['sources'].keys())
print('Available source databases:')
for db in source_databases:
    print(f'  - {db}')

Available source databases:
  - BAPS
  - CHVD
  - ELGV
  - GOV2
  - GPD
  - GSV
  - GVD
  - HPGC
  - IGVD
  - MGV
  - MetaVR
  - OPD
  - OVD
  - PhagesDB
  - STV
  - SVD
  - TYMEFLIES
  - TemPhD
  - UHGV
  - URPC
  - VMGC


In [5]:
# List phages from a specific source
source_to_list = 'PhagesDB'  # Change this to explore other sources
phages_in_source = gff3.list_phages(source_db=source_to_list)

print(f'Phages in {source_to_list}: {len(phages_in_source):,}')
print(f'\nFirst 10 phages:')
for pid in sorted(phages_in_source)[:10]:
    print(f'  {pid}')

Phages in PhagesDB: 3,754

First 10 phages:
  Actinoplanes_phage_phiAsp2
  Arthrobacter_Hestia_complete
  Arthrobacter_phage_Abba
  Arthrobacter_phage_Abidatro
  Arthrobacter_phage_Adaia
  Arthrobacter_phage_Adat
  Arthrobacter_phage_Adolin
  Arthrobacter_phage_Adumb2043
  Arthrobacter_phage_Aledel
  Arthrobacter_phage_Amelia


---
## 4. Retrieve GFF3 for a Single Phage

The `get_gff3()` method returns the raw GFF3 content for a phage as a string.
This preserves the full annotation structure including comments and metadata.

In [12]:
# Pick a phage to retrieve (use one from the list above)
if phages_in_source:
    sample_phage = sorted(phages_in_source)[0]
else:
    sample_phage = None


if sample_phage:
    print(f'Retrieving GFF3 for: {sample_phage}')
    print(f'Source database: {gff3.get_source_db(sample_phage)}')
    print()
    
    content = gff3.get_gff3(sample_phage)
    print(f'GFF3 content ({len(content):,} bytes):')
    print('-' * 60)
    # Show first 30 lines
    for i, line in enumerate(content.split('\n')[:30]):
        print(line)
    if len(content.split('\n')) > 30:
        print(f'... ({len(content.split(chr(10)))} total lines)')
else:
    print('No phages available. Run the pipeline first.')

GFF3 file not found: /data/processed/gff3/PhagesDB.gff3


Retrieving GFF3 for: Actinoplanes_phage_phiAsp2
Source database: PhagesDB

GFF3 content (0 bytes):
------------------------------------------------------------



---
## 5. Memory-Efficient Line Iterator

For large GFF3 files, use `get_gff3_lines()` to iterate line by line
without loading the entire content into memory.

In [7]:
if sample_phage:
    print(f'Iterating GFF3 lines for: {sample_phage}')
    print()
    
    line_count = 0
    cds_count = 0
    for line in gff3.get_gff3_lines(sample_phage):
        line_count += 1
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        if len(fields) >= 9 and fields[2] == 'CDS':
            cds_count += 1
            if cds_count <= 5:  # Show first 5 CDS
                print(f'CDS: {fields[0]}:{fields[3]}-{fields[4]} ({fields[6]})')
    
    print(f'\nTotal lines: {line_count}')
    print(f'CDS features: {cds_count}')

GFF3 file not found: /data/processed/gff3/PhagesDB.gff3


Iterating GFF3 lines for: Actinoplanes_phage_phiAsp2


Total lines: 0
CDS features: 0


---
## 6. Parse GFF3 into a DataFrame

Parse GFF3 lines into a structured pandas DataFrame for analysis.
The standard GFF3 columns are: `seqid`, `source`, `type`, `start`, `end`,
`score`, `strand`, `phase`, `attributes`.

In [10]:
def parse_gff3_attributes(attr_string):
    """Parse GFF3 attributes column into a dictionary."""
    attrs = {}
    if not attr_string or attr_string == '.':
        return attrs
    for pair in attr_string.split(';'):
        if '=' in pair:
            key, value = pair.split('=', 1)
            attrs[key.strip()] = value.strip()
    return attrs


def gff3_to_dataframe(gff3_content):
    """Parse GFF3 content into a DataFrame."""
    rows = []
    for line in gff3_content.split('\n'):
        if line.startswith('#') or not line.strip():
            continue
        fields = line.strip().split('\t')
        if len(fields) >= 9:
            attrs = parse_gff3_attributes(fields[8])
            rows.append({
                'seqid': fields[0],
                'source': fields[1],
                'type': fields[2],
                'start': int(fields[3]),
                'end': int(fields[4]),
                'score': fields[5],
                'strand': fields[6],
                'phase': fields[7],
                'attributes': fields[8],
                'ID': attrs.get('ID', ''),
                'product': attrs.get('product', ''),
                'protein_id': attrs.get('protein_id', ''),
            })
    return pd.DataFrame(rows)


if sample_phage:
    content = gff3.get_gff3(sample_phage)
    print(content)
    df = gff3_to_dataframe(content)
    print(f'Parsed GFF3 for: {sample_phage}')
    print(f'Total features: {len(df)}')
    print(f'\nFeature types:')
    print(df['type'].value_counts().to_string())
    print(f'\nCDS features (first 10):')
    df_cds = df[df['type'] == 'CDS'].head(10)
    display(df_cds[['seqid', 'start', 'end', 'strand', 'ID', 'product']])

GFF3 file not found: /data/processed/gff3/PhagesDB.gff3



Parsed GFF3 for: Actinoplanes_phage_phiAsp2
Total features: 0

Feature types:


KeyError: 'type'

---
## 7. Gene Length Distribution

Analyze the distribution of gene lengths from GFF3 annotations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

if sample_phage and len(df) > 0:
    df['length'] = df['end'] - df['start'] + 1
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gene length distribution
    cds_lengths = df[df['type'] == 'CDS']['length']
    if len(cds_lengths) > 0:
        axes[0].hist(cds_lengths, bins=30, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Gene Length (bp)')
        axes[0].set_ylabel('Count')
        axes[0].set_title(f'CDS Length Distribution - {sample_phage}')
        axes[0].axvline(cds_lengths.mean(), color='red', linestyle='--', 
                       label=f'Mean: {cds_lengths.mean():.0f} bp')
        axes[0].legend()
    
    # Strand distribution
    strand_counts = df[df['type'] == 'CDS']['strand'].value_counts()
    if len(strand_counts) > 0:
        axes[1].pie(strand_counts.values, labels=strand_counts.index, 
                   autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Strand Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Save figure
    fig_path = results_dir / f'gene_distribution_{sample_phage}.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Figure saved: {fig_path}')

---
## 8. Check if a Phage Exists

Use `has_phage()` to quickly check if a phage has GFF3 annotations
without loading the full content.

In [ ]:
# Check specific phages
test_phages = [
    'Mycobacterium_phage_NuevoMundo',
    'Escherichia_phage_T4',
    'Nonexistent_phage_test',
]

for phage_id in test_phages:
    exists = gff3.has_phage(phage_id)
    status = 'FOUND' if exists else 'NOT FOUND'
    print(f'{phage_id}: {status}')

---
## 9. Integration with Phage Metadata

Combine GFF3 annotations with phage metadata from the DuckDB database
to get a complete picture.

In [ ]:
import pbi

# Connect to the main database
try:
    retriever = pbi.quick_connect()
    print('Connected to database')
    
    # Get phage metadata
    phage_metadata = retriever.get_phage_metadata(limit=100)
    print(f'Retrieved metadata for {len(phage_metadata)} phages')
    
    # Check which phages have GFF3 annotations
    phage_ids = phage_metadata['Phage_ID'].tolist()
    has_gff3 = [gff3.has_phage(pid) for pid in phage_ids]
    phage_metadata['has_gff3'] = has_gff3
    
    print(f'\nPhages with GFF3: {sum(has_gff3)} / {len(has_gff3)}')
    display(phage_metadata[['Phage_ID', 'Source_DB', 'Length', 'has_gff3']].head(10))
    
    retriever.close()
except Exception as e:
    print(f'Database connection failed: {e}')
    print('This example requires the full PBI pipeline to have been run.')

---
## 10. Batch Retrieval Example

Retrieve GFF3 annotations for multiple phages and combine them.

In [ ]:
# Get phages from a specific source
source_for_batch = 'PhagesDB'
phages_for_batch = gff3.list_phages(source_db=source_for_batch)[:5]  # Limit to 5 for demo

print(f'Batch retrieving GFF3 for {len(phages_for_batch)} phages from {source_for_batch}:')
print()

all_cds = []
for phage_id in phages_for_batch:
    content = gff3.get_gff3(phage_id)
    if content:
        df_phage = gff3_to_dataframe(content)
        df_cds = df_phage[df_phage['type'] == 'CDS'].copy()
        df_cds['phage_id'] = phage_id
        all_cds.append(df_cds)
        print(f'  {phage_id}: {len(df_cds)} CDS features')

if all_cds:
    combined_df = pd.concat(all_cds, ignore_index=True)
    print(f'\nTotal CDS across all phages: {len(combined_df)}')
    print(f'\nSample products:')
    products = combined_df[combined_df['product'] != '']['product'].head(10)
    for prod in products:
        print(f'  - {prod}')

---
## Summary

### Key Takeaways

1. **`GFF3Retriever`** provides O(1) access to phage gene annotations
2. **Raw GFF3** is preserved for full fidelity; parse as needed
3. **Integration** with DuckDB metadata enables combined analysis
4. **Memory-efficient** line iterator for large files

### API Reference

| Method | Description |
|--------|-------------|
| `get_gff3(phage_id)` | Return full GFF3 content as string |
| `get_gff3_lines(phage_id)` | Yield GFF3 lines (memory-efficient) |
| `list_phages(source_db=)` | List all phage IDs, optionally filtered |
| `has_phage(phage_id)` | Check if phage exists in index |
| `get_source_db(phage_id)` | Get source database for a phage |
| `stats()` | Get index statistics |

### Next Steps

- Parse GFF3 to extract specific protein products
- Build gene presence/absence matrices across phages
- Integrate with protein sequences from `SequenceRetriever`